In [0]:
dbutils.widgets.text('p_batch_id', '')
v_batch_id = dbutils.widgets.get('p_batch_id')

In [0]:
%run ../common/environmental_config

In [0]:
%run ../common/00.bronze-helper

In [0]:
source_file = f"{loading_folder_path}/{v_batch_id}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

In [0]:
#schema
from pyspark.sql.types import StructType, StructField, StringType, DateType

name_schema = StructType([
    StructField('givenName', StringType()),
    StructField("familyName", StringType())
])

drivers_schema = StructType([
    StructField("driverId", StringType()),
    StructField("name" , name_schema),
    StructField("dateOfBirth", DateType()),
    StructField("nationality", StringType()),
    StructField("url", StringType())
])


In [0]:
df_drivers = (
    spark.read
         .format('json')
         .schema(drivers_schema)
         .option('mode', 'failfast')
         .load(source_file)
)

In [0]:
final_df_drivers = add_ingestion_metadata(df_drivers)

In [0]:
write_to_bronze (input_df = final_df_drivers,
                 table_name = table_name,
                 batch_id = v_batch_id)